## 论文详解：Generative Modeling by Estimating Gradients of the Data Distribution  
论文： Generative Modeling by Estimating Gradients of the Data Distribution  
作者： Yang Song、Stefano Ermon  
会议： NeurIPS 2019  
论文： arXiv:1907.05600  
这篇论文是现代 Score-based 生成模型的重要奠基工作之一，提出了 Noise Conditional Score Network（NCSN，噪声条件分数网络），以及 Annealed Langevin Dynamics（退火朗之万动力学）。  

先明确一个重要区别：  
这篇论文还没有提出连续时间 VPSDE，也没有从 ELBO 出发推导训练损失。  
它的核心思想是：  
不直接学习数据的概率密度，而是学习概率密度的梯度；然后利用这个梯度，从随机噪声逐渐生成真实数据。  

整篇论文的理论路线是：
$$
\boxed{
\begin{aligned}
&\text{真实数据分布 }p_{\rm data}(x)\\
&\Downarrow\\
&\text{学习 Score：}\nabla_x\log p_{\rm data}(x)\\
&\Downarrow\\
&\text{发现直接学习 Score 存在两个困难}\\
&\Downarrow\\
&\text{对数据添加多个尺度的高斯噪声}\\
&\Downarrow\\
&\text{使用 DSM 学习各噪声分布的 Score}\\
&\Downarrow\\
&\text{使用 Annealed Langevin Dynamics 采样}\\
&\Downarrow\\
&\text{逐渐从噪声分布恢复数据分布}
\end{aligned}
}
$$


## 一、论文的研究问题与整体思路

这篇论文的核心思想可以概括为：

> **通过 Score Matching 学习多个高斯扰动数据分布的 Score，再利用退火 Langevin Dynamics，从随机噪声中逐渐生成真实数据。**

它最重要的创新不是提出一个新的神经网络，而是将两个部分结合起来：

- **Noise Conditional Score Network（NCSN）**：学习不同噪声尺度下的概率密度梯度。
- **Annealed Langevin Dynamics**：利用不同噪声尺度的 Score，逐渐将随机样本转化为真实数据样本。

作者首先提出一个问题：

假设真实数据分布为：

$$
p_{\mathrm{data}}(x)
$$

我们能否不直接学习概率密度，而是学习：

$$
\boxed{
s(x)=\nabla_x\log p_{\mathrm{data}}(x)
}
$$

然后仅依赖这个梯度场生成数据？

理论上可以，因为 Langevin Dynamics 只需要概率分布的 Score，就能够在适当条件下从该分布采样。

但作者发现，直接这样做存在两个困难：

1. 真实数据可能集中在低维流形上，导致原始 Score 难以定义和估计。
2. 低概率区域缺少训练样本，Score 预测不准确，同时不同模态之间的 Langevin 采样混合速度很慢。

论文提出的解决方案是：

$$
\boxed{
\begin{aligned}
p_{\mathrm{data}}(x)
&\xrightarrow{\text{多个尺度的高斯扰动}}
\{q_{\sigma_i}(x)\}_{i=1}^{L}\\
&\xrightarrow{\text{Denoising Score Matching}}
s_\theta(x,\sigma_i)\\
&\xrightarrow{\text{Annealed Langevin Dynamics}}
x_{\mathrm{generated}}
\end{aligned}
}
$$

下面按照论文的实际顺序，逐步分析每一个环节，并详细推导核心公式。

---

# 二、Background：Score Matching 的数学基础

对应论文第 2.1 节。

## 2.1 什么是 Score？

假设：

$$
x\in\mathbb R^D
$$

服从概率密度：

$$
p(x)
$$

定义 Score：

$$
\boxed{
s(x)=\nabla_x\log p(x)
}
$$

其中：

- $D$：数据维度。
- $p(x)$：概率密度。
- $\nabla_x$：对数据变量 $x$ 求梯度。
- $s(x)\in\mathbb R^D$：Score 向量。

Score 的每个分量为：

$$
s_j(x)=\frac{\partial\log p(x)}{\partial x_j}
$$

因此，Score 描述了对数概率密度在当前位置的变化方向。

例如，一维高斯分布：

$$
p(x)=\mathcal N(x;\mu,\sigma^2)
$$

有：

$$
\log p(x)
=
-\frac{(x-\mu)^2}{2\sigma^2}
-\frac12\log(2\pi\sigma^2)
$$

对 $x$ 求导：

$$
\boxed{
s(x)=-\frac{x-\mu}{\sigma^2}
}
$$

当：

$$
x>\mu
$$

时：

$$
s(x)<0
$$

当：

$$
x<\mu
$$

时：

$$
s(x)>0
$$

所以 Score 指向概率密度增大的方向。

不过要注意，**Score 本身不是概率密度，也不是一个归一化的概率分布，而是一个向量场。**

论文使用神经网络：

$$
s_\theta:\mathbb R^D\rightarrow\mathbb R^D
$$

来拟合真实 Score：

$$
s_\theta(x)\approx\nabla_x\log p_{\mathrm{data}}(x)
$$

其中：

$$
\theta
$$

表示网络参数。

---

## 2.2 为什么不直接监督真实 Score？

最直接的训练目标是：

$$
\boxed{
\mathcal J(\theta)
=
\frac12
\mathbb E_{x\sim p_{\mathrm{data}}}
\left[
\left\|
s_\theta(x)-\nabla_x\log p_{\mathrm{data}}(x)
\right\|^2
\right]
}
$$

这相当于让网络输出的向量场尽可能接近真实 Score。

但问题是：

$$
p_{\mathrm{data}}(x)
$$

未知。

我们只有从该分布采样得到的数据：

$$
\{x_i\}_{i=1}^{N}
$$

因此：

$$
\nabla_x\log p_{\mathrm{data}}(x)
$$

也未知。

无法直接使用普通监督学习。

Score Matching 的关键在于：**通过数学变换，将损失中的真实 Score 消去。**

---

## 2.3 经典 Score Matching 的完整推导

定义：

$$
s^*(x)=\nabla_x\log p_{\mathrm{data}}(x)
$$

原始目标：

$$
\mathcal J(\theta)
=
\frac12\mathbb E_p
\left[
\|s_\theta(x)-s^*(x)\|^2
\right]
$$

其中为了简洁：

$$
p=p_{\mathrm{data}}
$$

展开平方：

$$
\begin{aligned}
\mathcal J(\theta)
={}&
\frac12\mathbb E_p\|s_\theta(x)\|^2\\
&-\mathbb E_p[s_\theta(x)^\top s^*(x)]\\
&+\frac12\mathbb E_p\|s^*(x)\|^2
\end{aligned}
$$

最后一项：

$$
\frac12\mathbb E_p\|s^*(x)\|^2
$$

不依赖 $\theta$。

因此可以记为常数：

$$
C
$$

于是：

$$
\mathcal J(\theta)
=
\frac12\mathbb E_p\|s_\theta(x)\|^2
-\mathbb E_p[s_\theta(x)^\top s^*(x)]
+C
$$

接下来处理交叉项。

由于：

$$
s^*(x)=\frac{\nabla_xp(x)}{p(x)}
$$

所以：

$$
\begin{aligned}
\mathbb E_p[s_\theta^\top s^*]
&=
\int p(x)s_\theta(x)^\top
\frac{\nabla_xp(x)}{p(x)}dx\\
&=
\int s_\theta(x)^\top\nabla_xp(x)\,dx
\end{aligned}
$$

将向量写成各个维度：

$$
\int
\sum_{j=1}^{D}
s_{\theta,j}(x)
\frac{\partial p(x)}{\partial x_j}dx
$$

交换求和与积分：

$$
=
\sum_{j=1}^{D}
\int
s_{\theta,j}(x)
\frac{\partial p(x)}{\partial x_j}dx
$$

对第 $j$ 个维度分部积分：

$$
\int
s_{\theta,j}(x)
\frac{\partial p(x)}{\partial x_j}dx
=
-\int
p(x)
\frac{\partial s_{\theta,j}(x)}{\partial x_j}dx
$$

这里假设概率密度和网络满足相应的光滑性、可积性及边界条件，使分部积分的边界项为零。

于是：

$$
\begin{aligned}
\mathbb E_p[s_\theta^\top s^*]
&=
-\sum_{j=1}^{D}
\int
p(x)
\frac{\partial s_{\theta,j}(x)}{\partial x_j}dx\\
&=
-\mathbb E_p
\left[
\sum_{j=1}^{D}
\frac{\partial s_{\theta,j}(x)}{\partial x_j}
\right]
\end{aligned}
$$

定义 Score 网络的 Jacobian：

$$
J_\theta(x)
=
\nabla_xs_\theta(x)
$$

其迹为：

$$
\operatorname{tr}(J_\theta)
=
\sum_{j=1}^{D}
\frac{\partial s_{\theta,j}(x)}{\partial x_j}
$$

因此：

$$
\boxed{
\mathbb E_p[s_\theta^\top s^*]
=
-\mathbb E_p[\operatorname{tr}(J_\theta)]
}
$$

代入原始目标：

$$
\mathcal J(\theta)
=
\mathbb E_p
\left[
\frac12\|s_\theta(x)\|^2
+
\operatorname{tr}(J_\theta(x))
\right]
+C
$$

去掉与参数无关的常数：

$$
\boxed{
\mathcal J_{\mathrm{SM}}(\theta)
=
\mathbb E_{p_{\mathrm{data}}}
\left[
\operatorname{tr}(\nabla_xs_\theta(x))
+
\frac12\|s_\theta(x)\|^2
\right]
}
\tag{1}
$$

这就是论文的公式 (1)。

**推导的关键结果是：训练目标不再需要知道真实数据分布的 Score。**

但是，这个目标仍然存在计算困难。

对于：

$$
s_\theta:\mathbb R^D\rightarrow\mathbb R^D
$$

Jacobian 是一个：

$$
D\times D
$$

矩阵。

而损失需要计算：

$$
\operatorname{tr}(\nabla_xs_\theta)
$$

在高维数据和深度网络中，这可能带来较大的计算开销。论文因此讨论了 Denoising Score Matching 和 Sliced Score Matching 两种替代方案。[arXiv](https://arxiv.org/pdf/1907.05600?utm_source=chatgpt.com)

---

## 2.4 Sliced Score Matching 为什么能够避免完整 Jacobian？

论文公式 (3) 给出：

$$
\boxed{
\mathcal J_{\mathrm{SSM}}(\theta)
=
\mathbb E_{v}
\mathbb E_{x\sim p_{\mathrm{data}}}
\left[
v^\top J_\theta(x)v
+
\frac12\|s_\theta(x)\|^2
\right]
}
\tag{3}
$$

其中：

$$
v\sim p_v
$$

是一个随机向量。

例如：

$$
v\sim\mathcal N(0,I)
$$

为什么随机向量能够用于估计 Jacobian 的迹？

考虑：

$$
v^\top Jv
=
\sum_{i=1}^{D}\sum_{j=1}^{D}
v_iJ_{ij}v_j
$$

对随机向量取期望：

$$
\mathbb E_v[v^\top Jv]
=
\sum_{i,j}J_{ij}\mathbb E[v_iv_j]
$$

由于标准高斯随机向量满足：

$$
\mathbb E[v_iv_j]=
\begin{cases}
1,&i=j\\
0,&i\neq j
\end{cases}
$$

因此：

$$
\mathbb E_v[v^\top Jv]
=
\sum_iJ_{ii}
$$

最终：

$$
\boxed{
\mathbb E_v[v^\top Jv]
=
\operatorname{tr}(J)
}
$$

所以可以使用随机向量估计 Jacobian 的迹，而不必显式计算完整矩阵。

论文指出，SSM 能够直接估计原始数据分布的 Score，但相比 DSM 通常需要更多计算，因此最终主要采用 DSM。

---

# 三、Denoising Score Matching：论文训练方法的基础

对应论文第 2.1 节及第 4.2 节。

## 3.1 从原始数据分布构造扰动分布

假设：

$$
x\sim p_{\mathrm{data}}(x)
$$

我们人为添加高斯噪声：

$$
\boxed{
\widetilde x=x+\sigma\epsilon
}
$$

其中：

$$
\epsilon\sim\mathcal N(0,I)
$$

$$
\sigma>0
$$

是噪声标准差。

因此，给定原始数据 $x$ 时：

$$
\boxed{
q_\sigma(\widetilde x|x)
=
\mathcal N(\widetilde x;x,\sigma^2I)
}
$$

这里：

$$
q_\sigma(\widetilde x|x)
$$

是条件扰动分布。

但是，我们真正希望学习的是所有原始数据加噪后形成的边缘分布：

$$
\boxed{
q_\sigma(\widetilde x)
=
\int
p_{\mathrm{data}}(x)
q_\sigma(\widetilde x|x)dx
}
$$

这个积分相当于将原始数据分布与高斯分布做卷积：

$$
\boxed{
q_\sigma
=
p_{\mathrm{data}}*\mathcal N(0,\sigma^2I)
}
$$

因此：

$$
q_\sigma
$$

是经过高斯平滑的数据分布。

我们希望训练：

$$
s_\theta(\widetilde x,\sigma)
$$

使其满足：

$$
\boxed{
s_\theta(\widetilde x,\sigma)
\approx
\nabla_{\widetilde x}\log q_\sigma(\widetilde x)
}
$$

但问题仍然存在：

$$
q_\sigma(\widetilde x)
$$

需要对真实数据分布积分，通常无法直接计算。

所以它的 Score 也未知。

---

## 3.2 为什么条件分布的 Score 可以直接计算？

虽然边缘分布：

$$
q_\sigma(\widetilde x)
$$

难以计算，但条件分布：

$$
q_\sigma(\widetilde x|x)
$$

是已知高斯分布。

根据高斯密度公式：

$$
q_\sigma(\widetilde x|x)
=
\frac1{(2\pi\sigma^2)^{D/2}}
\exp\left[
-\frac{\|\widetilde x-x\|^2}{2\sigma^2}
\right]
$$

取对数：

$$
\log q_\sigma(\widetilde x|x)
=
-\frac D2\log(2\pi\sigma^2)
-\frac{\|\widetilde x-x\|^2}{2\sigma^2}
$$

对 $\widetilde x$ 求梯度：

$$
\boxed{
\nabla_{\widetilde x}\log q_\sigma(\widetilde x|x)
=
-\frac{\widetilde x-x}{\sigma^2}
}
$$

又因为：

$$
\widetilde x=x+\sigma\epsilon
$$

所以：

$$
\widetilde x-x=\sigma\epsilon
$$

最终：

$$
\boxed{
\nabla_{\widetilde x}\log q_\sigma(\widetilde x|x)
=
-\frac{\epsilon}{\sigma}
}
$$

因此，我们可以直接得到监督目标：

$$
-\frac{\epsilon}{\sigma}
$$

不需要知道真实数据分布。

于是构造：

$$
\boxed{
\ell(\theta;\sigma)
=
\frac12
\mathbb E_{x\sim p_{\mathrm{data}},\,\epsilon\sim\mathcal N(0,I)}
\left[
\left\|
s_\theta(x+\sigma\epsilon,\sigma)
+
\frac{\epsilon}{\sigma}
\right\|^2
\right]
}
\tag{5}
$$

这就是论文公式 (5) 的等价写法。

但这里有一个非常关键的问题：

**我们训练时监督的是条件 Score，为什么最终能够学习边缘 Score？**

这需要进一步证明。

---

## 3.3 严格推导：DSM 为什么能够学习边缘 Score？

定义条件 Score：

$$
u(\widetilde x,x)
=
\nabla_{\widetilde x}\log q_\sigma(\widetilde x|x)
$$

定义真实边缘 Score：

$$
s^*(\widetilde x)
=
\nabla_{\widetilde x}\log q_\sigma(\widetilde x)
$$

为了简洁，这一节暂时省略固定的 $\sigma$。

我们的 DSM 目标为：

$$
\mathcal J_{\mathrm{DSM}}
=
\frac12
\mathbb E_{x,\widetilde x}
\left[
\|s_\theta(\widetilde x)-u(\widetilde x,x)\|^2
\right]
$$

首先证明：

$$
\boxed{
\mathbb E[u(\widetilde x,x)|\widetilde x]
=
s^*(\widetilde x)
}
$$

从边缘分布出发：

$$
q_\sigma(\widetilde x)
=
\int
p_{\mathrm{data}}(x)
q_\sigma(\widetilde x|x)dx
$$

对 $\widetilde x$ 求梯度：

$$
\nabla_{\widetilde x}q_\sigma(\widetilde x)
=
\int
p_{\mathrm{data}}(x)
\nabla_{\widetilde x}q_\sigma(\widetilde x|x)dx
$$

利用：

$$
\nabla q=q\nabla\log q
$$

得到：

$$
\begin{aligned}
\nabla_{\widetilde x}q_\sigma(\widetilde x)
=
\int
&p_{\mathrm{data}}(x)
q_\sigma(\widetilde x|x)\\
&\cdot\nabla_{\widetilde x}\log q_\sigma(\widetilde x|x)dx
\end{aligned}
$$

两边除以：

$$
q_\sigma(\widetilde x)
$$

于是：

$$
\begin{aligned}
\nabla_{\widetilde x}\log q_\sigma(\widetilde x)
=
\int
&\frac{
p_{\mathrm{data}}(x)q_\sigma(\widetilde x|x)
}{
q_\sigma(\widetilde x)
}\\
&\cdot u(\widetilde x,x)dx
\end{aligned}
$$

根据贝叶斯公式：

$$
\frac{
p_{\mathrm{data}}(x)q_\sigma(\widetilde x|x)
}{
q_\sigma(\widetilde x)
}
=
q_\sigma(x|\widetilde x)
$$

因此：

$$
\begin{aligned}
\nabla_{\widetilde x}\log q_\sigma(\widetilde x)
&=
\int q_\sigma(x|\widetilde x)
u(\widetilde x,x)dx\\
&=
\mathbb E[u(\widetilde x,x)|\widetilde x]
\end{aligned}
$$

所以：

$$
\boxed{
s^*(\widetilde x)
=
\mathbb E[u(\widetilde x,x)|\widetilde x]
}
$$

接下来分析 DSM Loss。

在平方误差中加减边缘 Score：

$$
s_\theta-u
=
(s_\theta-s^*)+(s^*-u)
$$

展开：

$$
\begin{aligned}
\|s_\theta-u\|^2
={}&
\|s_\theta-s^*\|^2\\
&+\|s^*-u\|^2\\
&+2(s_\theta-s^*)^\top(s^*-u)
\end{aligned}
$$

对联合分布取期望：

$$
\mathbb E_{x,\widetilde x}
$$

其中交叉项为：

$$
\mathbb E[
(s_\theta-s^*)^\top(s^*-u)
]
$$

先对给定 $\widetilde x$ 的条件分布取期望：

$$
\begin{aligned}
&\mathbb E[
(s_\theta-s^*)^\top(s^*-u)
]\\
&=
\mathbb E_{\widetilde x}
\left[
(s_\theta-s^*)^\top
\mathbb E[s^*-u|\widetilde x]
\right]
\end{aligned}
$$

根据前面的恒等式：

$$
\mathbb E[u|\widetilde x]=s^*
$$

因此：

$$
\mathbb E[s^*-u|\widetilde x]=0
$$

交叉项消失。

最终：

$$
\boxed{
\begin{aligned}
\mathbb E_{x,\widetilde x}
\|s_\theta-u\|^2
={}&
\mathbb E_{\widetilde x}
\|s_\theta-s^*\|^2\\
&+
\mathbb E_{x,\widetilde x}
\|u-s^*\|^2
\end{aligned}
}
$$

第二项与模型参数无关。

因此：

$$
\boxed{
\mathcal J_{\mathrm{DSM}}(\theta)
=
\frac12
\mathbb E_{\widetilde x\sim q_\sigma}
\left[
\|s_\theta(\widetilde x)-\nabla\log q_\sigma(\widetilde x)\|^2
\right]
+C
}
$$

其中：

$$
C
$$

不依赖 $\theta$。

所以在模型表达能力充分、优化达到全局最优等条件下：

$$
\boxed{
s_\theta^*(\widetilde x,\sigma)
=
\nabla_{\widetilde x}\log q_\sigma(\widetilde x)
}
$$

这就严格证明了 DSM 的正确性。

**关键理解：每次训练的监督目标虽然是某个干净样本对应的条件高斯 Score，但对所有可能的干净样本取条件平均后，最优预测恰好就是加噪边缘分布的 Score。**

这也是整篇论文最重要的数学基础之一。[arXiv](https://arxiv.org/pdf/1907.05600?utm_source=chatgpt.com)

# 四、Langevin Dynamics：为什么知道 Score 就能生成数据？

对应论文第 2.2 节，公式 (4)。

前面解决了如何训练 Score 网络的问题。

但网络只输出：

$$
s_\theta(x)\approx\nabla_x\log p(x)
$$

它没有直接输出生成样本。

那么如何仅依靠一个概率密度的梯度场，从该分布中采样？

作者采用 **Langevin Dynamics（朗之万动力学）**。

## 4.1 Langevin Dynamics 的基本公式

论文公式 (4) 为：

$$
\boxed{
x_{k+1}
=
x_k+
\frac{\eta}{2}\nabla_x\log p(x_k)
+
\sqrt{\eta}\,z_k
}
\tag{4}
$$

其中：

- $x_k$：第 $k$ 次迭代的样本。
- $\eta>0$：采样步长。
- $z_k\sim\mathcal N(0,I)$：独立高斯噪声。
- $p(x)$：希望采样的目标分布。

公式包含两部分：

$$
\underbrace{\frac{\eta}{2}\nabla\log p(x_k)}_{\text{Score 引导}}
+
\underbrace{\sqrt{\eta}z_k}_{\text{随机探索}}
$$

为什么这样的迭代能够从 $p(x)$ 中采样？

需要从连续时间随机过程开始推导。

## 4.2 构造连续 Langevin SDE

考虑随机微分方程：

$$
\boxed{
dX_\tau
=
\frac12\nabla_x\log p(X_\tau)d\tau
+
dW_\tau
}
$$

这里：

- $\tau$：Langevin 动力学运行时间。
- $W_\tau$：标准布朗运动。
- $p(x)$：目标概率密度。

需要强调，$\tau$ 是采样动力学的时间参数，不是噪声尺度的索引。

这个 SDE 的漂移为：

$$
b(x)=\frac12\nabla\log p(x)
$$

扩散系数为：

$$
g=1
$$

我们希望证明：

$$
p(x)
$$

是该随机过程的平稳分布。

### 第一步：写出 Fokker–Planck 方程

对于一般 SDE：

$$
dX_\tau=b(X_\tau)d\tau+dW_\tau
$$

设其在时间 $\tau$ 的概率密度为：

$$
\rho_\tau(x)
$$

那么 Fokker–Planck 方程为：

$$
\boxed{
\frac{\partial\rho_\tau}{\partial\tau}
=
-\nabla\cdot[b(x)\rho_\tau(x)]
+
\frac12\Delta\rho_\tau(x)
}
$$

其中：

$$
\nabla\cdot
$$

是散度算子。

$$
\Delta
=
\sum_{j=1}^{D}
\frac{\partial^2}{\partial x_j^2}
$$

是拉普拉斯算子。

### 第二步：代入 Langevin 漂移

因为：

$$
b(x)=\frac12\nabla\log p(x)
$$

所以：

$$
\frac{\partial\rho_\tau}{\partial\tau}
=
-\frac12
\nabla\cdot
\left[
\rho_\tau\nabla\log p
\right]
+
\frac12\Delta\rho_\tau
$$

### 第三步：验证目标分布是平稳分布

如果：

$$
\rho_\tau(x)=p(x)
$$

则：

$$
p(x)\nabla\log p(x)
=
p(x)\frac{\nabla p(x)}{p(x)}
=
\nabla p(x)
$$

因此：

$$
\begin{aligned}
\frac{\partial\rho_\tau}{\partial\tau}
&=
-\frac12\nabla\cdot(\nabla p)
+
\frac12\Delta p\\
&=
-\frac12\Delta p+\frac12\Delta p\\
&=0
\end{aligned}
$$

所以：

$$
\boxed{
\frac{\partial\rho_\tau}{\partial\tau}=0
}
$$

说明：

$$
\boxed{
\rho_\tau(x)=p(x)
}
$$

是 Langevin SDE 的平稳解。

在满足遍历性等条件时，随着：

$$
\tau\rightarrow\infty
$$

随机过程的分布可以收敛到：

$$
p(x)
$$

这就是只依靠 Score 进行采样的理论基础。

## 4.3 从 Langevin SDE 推导离散采样公式

连续 SDE：

$$
dX_\tau
=
\frac12\nabla\log p(X_\tau)d\tau+dW_\tau
$$

在时间区间：

$$
[\tau_k,\tau_k+\eta]
$$

上积分：

$$
X_{\tau_k+\eta}-X_{\tau_k}
=
\frac12
\int_{\tau_k}^{\tau_k+\eta}
\nabla\log p(X_\tau)d\tau
+
W_{\tau_k+\eta}-W_{\tau_k}
$$

使用 Euler–Maruyama 近似：

$$
\int_{\tau_k}^{\tau_k+\eta}
\nabla\log p(X_\tau)d\tau
\approx
\eta\nabla\log p(X_{\tau_k})
$$

根据布朗运动增量性质：

$$
W_{\tau_k+\eta}-W_{\tau_k}
\sim
\mathcal N(0,\eta I)
$$

因此可以写成：

$$
W_{\tau_k+\eta}-W_{\tau_k}
=
\sqrt{\eta}z_k
$$

其中：

$$
z_k\sim\mathcal N(0,I)
$$

于是：

$$
\boxed{
x_{k+1}
=
x_k+
\frac{\eta}{2}\nabla\log p(x_k)
+
\sqrt{\eta}z_k
}
$$

这就得到了论文公式 (4)。

如果使用神经网络预测的 Score：

$$
s_\theta(x)\approx\nabla\log p(x)
$$

那么采样公式变为：

$$
\boxed{
x_{k+1}
=
x_k+
\frac{\eta}{2}s_\theta(x_k)
+
\sqrt{\eta}z_k
}
$$

## 4.4 为什么必须保留随机噪声？

考虑一个简单的一维高斯分布：

$$
p(x)=\mathcal N(0,1)
$$

它的 Score 为：

$$
\nabla\log p(x)=-x
$$

代入 Langevin Dynamics：

$$
\boxed{
x_{k+1}
=
\left(1-\frac{\eta}{2}\right)x_k
+
\sqrt{\eta}z_k
}
$$

如果去掉随机项：

$$
x_{k+1}
=
\left(1-\frac{\eta}{2}\right)x_k
$$

对于：

$$
0<\eta<4
$$

样本最终会收敛到：

$$
x=0
$$

也就是高斯分布的均值。

但这显然不是从：

$$
\mathcal N(0,1)
$$

采样。

真正的高斯分布需要保留方差。

现在保留随机项：

$$
x_{k+1}
=
ax_k+\sqrt{\eta}z_k
$$

其中：

$$
a=1-\frac{\eta}{2}
$$

假设：

$$
x_k
$$

在平稳状态下的方差为：

$$
V
$$

那么：

$$
V=a^2V+\eta
$$

解得：

$$
V=\frac{\eta}{1-a^2}
$$

代入：

$$
a=1-\frac{\eta}{2}
$$

得到：

$$
\begin{aligned}
V
&=
\frac{\eta}
{1-(1-\eta/2)^2}\\
&=
\frac{\eta}
{\eta-\eta^2/4}\\
&=
\frac1{1-\eta/4}
\end{aligned}
$$

因此：

$$
\boxed{
V=\frac1{1-\eta/4}
}
$$

当：

$$
\eta\rightarrow0
$$

时：

$$
V\rightarrow1
$$

这说明：

- 随机噪声保证采样结果具有非零方差。
- 步长越小，离散采样越接近连续 Langevin SDE。
- 有限步长会产生离散化偏差。

因此，**Score 引导和随机噪声共同决定了目标采样分布，不能只保留其中一个。**

论文也明确指出，有限步长和有限采样次数通常会带来误差；若要严格消除离散化偏差，可以使用 Metropolis–Hastings 校正，但本文实际采样时忽略了该校正。[arXiv](https://arxiv.org/pdf/1907.05600?utm_source=chatgpt.com)

---

# 五、直接使用 Score Matching 为什么会失败？

对应论文第 3 节。

这是作者提出 NCSN 的直接动机。

论文指出两个主要困难：

**第一个是低维流形问题，第二个是低概率区域问题。**

第二个问题又进一步影响 Score 估计和 Langevin 采样混合。

## 5.1 低维流形假设

假设数据空间为：

$$
\mathbb R^D
$$

但真实数据只集中在一个维度为：

$$
d<D
$$

的流形：

$$
\mathcal M\subset\mathbb R^D
$$

例如二维空间中的单位圆：

$$
\mathcal M
=
\{(x_1,x_2):x_1^2+x_2^2=1\}
$$

虽然：

$$
x\in\mathbb R^2
$$

但数据只能在圆周上变化。

因此数据实际只有一个自由度。

如果概率分布严格集中在圆周上，那么相对于二维 Lebesgue 测度，无法定义普通的二维概率密度：

$$
p_{\mathrm{data}}(x)
$$

因此：

$$
\nabla_x\log p_{\mathrm{data}}(x)
$$

也不能作为整个二维空间上的普通 Score 来处理。

而 Score Matching 正是在整个数据空间中学习：

$$
s_\theta:\mathbb R^D\rightarrow\mathbb R^D
$$

因此会遇到理论上的困难。

作者指出，经典 Score Matching 的一致性需要相应的分布支撑集和正则性条件，而低维流形上的分布可能不满足这些条件。

### 为什么高斯扰动能够解决？

定义：

$$
\widetilde x=x+\sigma\epsilon
$$

其中：

$$
\sigma>0
$$

则：

$$
q_\sigma(\widetilde x)
=
\int
p_{\mathrm{data}}(x)
\mathcal N(\widetilde x;x,\sigma^2I)dx
$$

由于高斯密度在整个空间严格为正：

$$
\mathcal N(\widetilde x;x,\sigma^2I)>0
$$

所以：

$$
\boxed{
q_\sigma(\widetilde x)>0
}
$$

对所有：

$$
\widetilde x\in\mathbb R^D
$$

成立。

而且高斯卷积具有平滑作用。

因此，即使原始分布集中在低维流形上，扰动后的分布也能够在整个空间中定义平滑概率密度。

于是：

$$
\boxed{
\nabla_{\widetilde x}\log q_\sigma(\widetilde x)
}
$$

可以正常定义。

这就是高斯扰动的第一个作用：**使 Score 估计在数学上更加良好。**

### 论文 Figure 1

作者在 CIFAR-10 上训练 Score 网络。

第一组实验不添加噪声。

第二组实验添加：

$$
\mathcal N(0,0.0001I)
$$

的微小高斯噪声。

结果表明：

- 不加噪声时，SSM 损失下降后出现明显的不规则波动。
- 添加微小噪声后，损失曲线更加稳定。

作者用这一实验支持低维流形问题会影响 Score 估计的判断。

但严格来说，这个实验是经验现象，并不能单独证明 CIFAR-10 的真实数据分布一定严格位于低维流形上。[arXiv](https://arxiv.org/pdf/1907.05600?utm_source=chatgpt.com)

---

## 5.2 低概率区域中的 Score 估计问题

考虑一个二维高斯混合分布：

$$
\boxed{
p(x)
=
\pi p_1(x)+(1-\pi)p_2(x)
}
$$

其中：

$$
p_1(x)=\mathcal N(x;\mu_1,I)
$$

$$
p_2(x)=\mathcal N(x;\mu_2,I)
$$

如果：

$$
\|\mu_1-\mu_2\|
$$

很大，那么两个模态之间存在低概率区域。

例如论文 Figure 2 使用：

$$
\boxed{
p(x)
=
\frac15\mathcal N((-5,-5),I)
+
\frac45\mathcal N((5,5),I)
}
$$

我们可以进一步推导这个混合分布的真实 Score。

### 第一步：求混合分布梯度

$$
\nabla p(x)
=
\pi\nabla p_1(x)
+
(1-\pi)\nabla p_2(x)
$$

对于单位协方差高斯：

$$
\nabla p_i(x)
=
p_i(x)(\mu_i-x)
$$

所以：

$$
\nabla p(x)
=
\pi p_1(x)(\mu_1-x)
+
(1-\pi)p_2(x)(\mu_2-x)
$$

### 第二步：计算 Score

$$
s(x)=\nabla\log p(x)
=
\frac{\nabla p(x)}{p(x)}
$$

代入：

$$
\boxed{
s(x)
=
\frac{
\pi p_1(x)(\mu_1-x)
+
(1-\pi)p_2(x)(\mu_2-x)
}{
\pi p_1(x)+(1-\pi)p_2(x)
}
}
$$

定义后验责任概率：

$$
r_1(x)
=
\frac{\pi p_1(x)}{p(x)}
$$

$$
r_2(x)
=
\frac{(1-\pi)p_2(x)}{p(x)}
$$

满足：

$$
r_1(x)+r_2(x)=1
$$

于是：

$$
\boxed{
s(x)
=
r_1(x)(\mu_1-x)
+
r_2(x)(\mu_2-x)
}
$$

也就是说：

**混合高斯分布的 Score，是各个高斯分量 Score 的后验加权平均。**

当 $x$ 靠近第一个模态时：

$$
r_1(x)\approx1
$$

因此：

$$
s(x)\approx\mu_1-x
$$

当 $x$ 靠近第二个模态时：

$$
r_2(x)\approx1
$$

因此：

$$
s(x)\approx\mu_2-x
$$

但是在两个模态之间，Score 的方向由两个高斯分量共同决定。

问题是：训练样本主要集中在两个模态附近。

两个模态之间的低概率区域很少出现训练数据。

因此，网络可能无法准确学习该区域的 Score。

论文 Figure 2 展示了这一现象：预测 Score 在高概率区域较准确，但在低概率区域与真实 Score 存在明显差异。[arXiv](https://arxiv.org/pdf/1907.05600?utm_source=chatgpt.com)

---

## 5.3 为什么不同模态之间的 Langevin 采样混合很慢？

这一部分对应论文第 3.2.2 节。

作者首先考虑两个支撑集不相交的分布：

$$
p_1(x),p_2(x)
$$

构造混合分布：

$$
\boxed{
p(x)=\pi p_1(x)+(1-\pi)p_2(x)
}
$$

其中：

$$
0<\pi<1
$$

假设：

$$
\operatorname{supp}(p_1)
\cap
\operatorname{supp}(p_2)
=
\varnothing
$$

在第一个分布的支撑区域内：

$$
p_2(x)=0
$$

所以：

$$
p(x)=\pi p_1(x)
$$

取对数：

$$
\log p(x)
=
\log\pi+\log p_1(x)
$$

对 $x$ 求梯度：

$$
\nabla\log p(x)
=
\nabla\log\pi+\nabla\log p_1(x)
$$

由于：

$$
\pi
$$

是常数：

$$
\nabla\log\pi=0
$$

因此：

$$
\boxed{
\nabla\log p(x)
=
\nabla\log p_1(x)
}
$$

同样，在第二个分布的支撑区域内：

$$
\boxed{
\nabla\log p(x)
=
\nabla\log p_2(x)
}
$$

所以：

$$
\boxed{
\text{两个模态内部的 Score 不包含混合权重 }\pi
}
$$

这意味着，仅仅依靠两个互不连通区域内部的 Score，无法确定两个模态之间的相对概率质量。

对于具有严格不相交支撑的分布，普通 Langevin Dynamics 还可能面临支撑区域之间无法正常穿越的问题。

对于实际中支撑集近似分离的分布，理论上仍可能发生模态切换，但往往需要非常长的采样时间。

### 为什么有限步 Langevin 采样容易产生错误的模态比例？

假设真实分布：

$$
p(x)=0.2p_1(x)+0.8p_2(x)
$$

但初始随机样本有：

$$
50\%
$$

落在第一个模态附近，另外：

$$
50\%
$$

落在第二个模态附近。

如果 Langevin Dynamics 很难跨越两个模态之间的低概率区域，那么：

- 第一个模态附近的样本长期停留在第一个模态。
- 第二个模态附近的样本长期停留在第二个模态。

最终得到的比例可能仍然接近：

$$
50\%:50\%
$$

而不是：

$$
20\%:80\%
$$

因此：

**即使使用完全准确的 Score，有限时间 Langevin 采样也可能无法恢复正确的模态权重。**

论文 Figure 3 使用真实 Score 进行了验证。

普通 Langevin Dynamics 产生了错误的模态比例，而退火 Langevin Dynamics 能够更好地恢复两个模态的相对概率质量。

这是论文最有说服力的实验之一，因为它说明：**问题不仅来自神经网络的 Score 预测误差，也来自采样动力学本身的混合速度。**[arXiv](https://arxiv.org/pdf/1907.05600?utm_source=chatgpt.com)

# 六、Noise Conditional Score Network：论文的核心模型

对应论文第 4.1 节。

前面已经发现，直接学习：

$$
\nabla\log p_{\mathrm{data}}(x)
$$

存在困难。

因此作者提出：

**对真实数据添加多个不同尺度的高斯噪声，并联合学习所有扰动分布的 Score。**

## 6.1 定义多个噪声尺度

选择：

$$
\boxed{
\sigma_1>\sigma_2>\cdots>\sigma_L>0
}
$$

其中：

$$
L
$$

表示噪声尺度数量。

论文采用几何级数：

$$
\boxed{
\frac{\sigma_1}{\sigma_2}
=
\frac{\sigma_2}{\sigma_3}
=
\cdots
=
\frac{\sigma_{L-1}}{\sigma_L}
>1
}
$$

因此：

$$
\boxed{
\sigma_i
=
\sigma_1
\left(
\frac{\sigma_L}{\sigma_1}
\right)^{\frac{i-1}{L-1}}
}
$$

每个噪声尺度对应一个扰动分布：

$$
\boxed{
q_{\sigma_i}(\widetilde x)
=
\int
p_{\mathrm{data}}(x)
\mathcal N(\widetilde x;x,\sigma_i^2I)dx
}
$$

当：

$$
\sigma_i
$$

较大时，原始数据分布被强烈平滑。

当：

$$
\sigma_i
$$

较小时，扰动分布更接近真实数据分布。

因此，我们得到一系列概率分布：

$$
q_{\sigma_1},
q_{\sigma_2},
\ldots,
q_{\sigma_L}
$$

作者希望学习：

$$
\boxed{
s_\theta(\widetilde x,\sigma_i)
\approx
\nabla_{\widetilde x}\log q_{\sigma_i}(\widetilde x)
}
$$

对于所有：

$$
i=1,\ldots,L
$$

都成立。

## 6.2 为什么只使用一个网络？

一种直接方案是训练：

$$
L
$$

个独立 Score 网络：

$$
s_{\theta_1}(x),\ldots,s_{\theta_L}(x)
$$

但这样需要：

$$
L
$$

套模型参数。

而不同噪声尺度下的 Score 存在明显关联。

因此作者使用一个共享参数的条件网络：

$$
\boxed{
s_\theta:\mathbb R^D\times\{\sigma_1,\ldots,\sigma_L\}
\rightarrow\mathbb R^D
}
$$

输入：

$$
(\widetilde x,\sigma_i)
$$

输出：

$$
s_\theta(\widetilde x,\sigma_i)
$$

这样网络可以根据噪声尺度，预测对应分布的 Score。

作者将其称为：

$$
\boxed{
\text{Noise Conditional Score Network（NCSN）}
}
$$

---

# 七、NCSN 的联合训练损失：公式 (5) 和 (6)

这是论文最重要的训练部分。

## 7.1 单个噪声尺度的 DSM

对于某个固定噪声尺度：

$$
\sigma_i
$$

加噪：

$$
\widetilde x=x+\sigma_i\epsilon
$$

条件 Score：

$$
\nabla_{\widetilde x}
\log q_{\sigma_i}(\widetilde x|x)
=
-\frac{\epsilon}{\sigma_i}
$$

所以：

$$
\boxed{
\ell(\theta;\sigma_i)
=
\frac12
\mathbb E_{x,\epsilon}
\left[
\left\|
s_\theta(x+\sigma_i\epsilon,\sigma_i)
+
\frac{\epsilon}{\sigma_i}
\right\|^2
\right]
}
\tag{5}
$$

这是一个普通的平方误差回归目标。

对于每个：

$$
\sigma_i
$$

它都有对应的最优 Score：

$$
s_\theta^*(\widetilde x,\sigma_i)
=
\nabla_{\widetilde x}\log q_{\sigma_i}(\widetilde x)
$$

## 7.2 多个噪声尺度的联合目标

将所有尺度的 DSM Loss 组合：

$$
\boxed{
\mathcal L(\theta)
=
\frac1L
\sum_{i=1}^{L}
\lambda(\sigma_i)\ell(\theta;\sigma_i)
}
\tag{6}
$$

其中：

$$
\lambda(\sigma_i)>0
$$

是每个噪声尺度的损失权重。

为什么要求：

$$
\lambda(\sigma_i)>0
$$

？

因为如果网络表达能力足够，并且每个尺度的损失都可以同时达到最优，那么正权重加权不会改变各个 DSM 目标的共同最优解。

即：

$$
\boxed{
s_\theta^*(x,\sigma_i)
=
\nabla_x\log q_{\sigma_i}(x)
}
$$

对所有：

$$
i=1,\ldots,L
$$

成立。

但是实际训练中，网络容量有限，且各个尺度共享参数。

所以权重会影响不同噪声尺度的优化程度。

因此，作者需要选择合适的：

$$
\lambda(\sigma_i)
$$

---

## 7.3 为什么作者选择 $\lambda(\sigma)=\sigma^2$？

论文的动机是：

**希望不同噪声尺度的损失贡献具有大致相同的数量级。**

对于高斯扰动：

$$
\widetilde x=x+\sigma\epsilon
$$

条件 Score 为：

$$
-\frac{\epsilon}{\sigma}
$$

因此：

$$
\left\|
\frac{\epsilon}{\sigma}
\right\|^2
=
\frac{\|\epsilon\|^2}{\sigma^2}
$$

因为：

$$
\epsilon\sim\mathcal N(0,I)
$$

所以：

$$
\mathbb E\|\epsilon\|^2=D
$$

于是：

$$
\boxed{
\mathbb E
\left\|
\frac{\epsilon}{\sigma}
\right\|^2
=
\frac D{\sigma^2}
}
$$

这说明条件 Score 的平方范数随着噪声尺度减小而增大。

如果所有尺度使用相同权重，小噪声尺度可能产生更大的损失数值和梯度。

论文进一步观察到，训练接近最优时，Score 的典型范数大致满足：

$$
\|s_\theta(x,\sigma)\|_2\propto\frac1\sigma
$$

于是作者选择：

$$
\boxed{
\lambda(\sigma)=\sigma^2
}
$$

这样可以抵消 Score 目标中：

$$
\frac1{\sigma^2}
$$

的主要尺度变化。

需要注意：这是一种基于尺度分析和经验观察的权重设计，并不是证明所有分布下的最优 Score 范数都严格等于 $c/\sigma$。

---

## 7.4 代入权重，得到最终训练损失

从：

$$
\mathcal L(\theta)
=
\frac1L
\sum_{i=1}^{L}
\sigma_i^2\ell(\theta;\sigma_i)
$$

开始。

代入公式 (5)：

$$
\begin{aligned}
\mathcal L(\theta)
=
\frac1{2L}
\sum_{i=1}^{L}
\sigma_i^2
\mathbb E
\left[
\left\|
s_\theta(\widetilde x,\sigma_i)
+
\frac{\epsilon}{\sigma_i}
\right\|^2
\right]
\end{aligned}
$$

利用：

$$
\sigma_i^2\|a\|^2=\|\sigma_i a\|^2
$$

得到：

$$
\boxed{
\mathcal L(\theta)
=
\frac1{2L}
\sum_{i=1}^{L}
\mathbb E_{x,\epsilon}
\left[
\left\|
\sigma_i s_\theta(x+\sigma_i\epsilon,\sigma_i)
+
\epsilon
\right\|^2
\right]
}
$$

这是论文实际采用的多尺度 DSM 目标的等价形式。

观察这个公式：

$$
\sigma_i s_\theta(\widetilde x,\sigma_i)
\approx-\epsilon
$$

因此，模型实际上在学习一个与添加噪声密切相关的向量场。

不过这里必须强调：

**网络在固定加噪输入下的最优输出，并不是每次添加的具体噪声 $-\epsilon/\sigma_i$，而是该条件目标对所有可能干净样本的条件期望。**

也就是：

$$
\boxed{
s_\theta^*(\widetilde x,\sigma_i)
=
\mathbb E
\left[
-\frac{\epsilon}{\sigma_i}
\;\middle|\;
\widetilde x
\right]
}
$$

而根据前面的 DSM 恒等式：

$$
\boxed{
s_\theta^*(\widetilde x,\sigma_i)
=
\nabla_{\widetilde x}
\log q_{\sigma_i}(\widetilde x)
}
$$

所以模型学习的是**加噪边缘分布的 Score**，而不是记忆每次训练时采样的随机噪声。

这一区别非常重要。[arXiv](https://arxiv.org/pdf/1907.05600?utm_source=chatgpt.com)

---

# 八、Annealed Langevin Dynamics：Algorithm 1 完整推导

对应论文第 4.3 节。

训练完成后，我们拥有：

$$
s_\theta(x,\sigma_i)
$$

对于每个：

$$
\sigma_i
$$

都能够预测对应扰动分布的 Score。

接下来需要利用这些 Score 生成样本。

## 8.1 为什么不能直接使用最小噪声尺度？

因为：

$$
\sigma_L
$$

很小。

所以：

$$
q_{\sigma_L}(x)
$$

非常接近原始数据分布。

虽然这样有利于生成真实数据，但也意味着它仍然具有复杂的多模态结构。

如果直接从随机初始化开始，使用：

$$
s_\theta(x,\sigma_L)
$$

运行 Langevin Dynamics，那么随机样本可能位于低概率区域。

而网络在这些区域的 Score 估计不一定准确。

同时，不同模态之间的概率屏障仍然可能导致采样混合缓慢。

因此，作者提出：

**先使用最大噪声尺度的 Score 进行采样，再逐渐降低噪声尺度。**

## 8.2 从大噪声分布逐渐过渡到小噪声分布

首先选择：

$$
\sigma_1>\sigma_2>\cdots>\sigma_L
$$

然后从一个简单初始分布采样：

$$
x^{(0)}\sim p_{\mathrm{init}}
$$

论文使用均匀随机噪声初始化。

第一阶段使用：

$$
s_\theta(x,\sigma_1)
$$

运行 Langevin Dynamics：

$$
x_{k+1}
=
x_k+
\frac{\eta_1}{2}s_\theta(x_k,\sigma_1)
+
\sqrt{\eta_1}z_k
$$

执行 $K$ 次。

得到：

$$
x^{(1)}
$$

理想情况下，它接近：

$$
q_{\sigma_1}
$$

然后第二阶段使用：

$$
s_\theta(x,\sigma_2)
$$

继续运行：

$$
x_{k+1}
=
x_k+
\frac{\eta_2}{2}s_\theta(x_k,\sigma_2)
+
\sqrt{\eta_2}z_k
$$

并将第一阶段的最终结果作为初始化。

如此反复：

$$
\boxed{
q_{\sigma_1}
\rightarrow
q_{\sigma_2}
\rightarrow
\cdots
\rightarrow
q_{\sigma_L}
}
$$

最终：

$$
q_{\sigma_L}\approx p_{\mathrm{data}}
$$

因此得到接近真实数据分布的生成样本。

注意，这里的箭头表示**有限步 Langevin 采样所希望实现的分布过渡**，并不表示存在一个解析的确定性映射，也不保证每个阶段都精确达到对应分布。

---

## 8.3 为什么 Langevin 步长要满足 $\eta_i\propto\sigma_i^2$？

论文采用：

$$
\boxed{
\eta_i
=
\varepsilon
\frac{\sigma_i^2}{\sigma_L^2}
}
$$

其中：

- $\varepsilon$：最小噪声尺度下的基础步长。
- $\eta_i$：第 $i$ 个噪声尺度的 Langevin 步长。

为什么这样设计？

考虑一次 Langevin 更新：

$$
\Delta x
=
\underbrace{
\frac{\eta_i}{2}s_\theta(x,\sigma_i)
}_{\text{Score 引导}}
+
\underbrace{
\sqrt{\eta_i}z
}_{\text{随机扰动}}
$$

作者希望不同噪声尺度下，引导项与随机项的相对强度大致稳定。

定义两者的平方范数比：

$$
R_i
=
\frac{
\left\|
\frac{\eta_i}{2}s_\theta(x,\sigma_i)
\right\|^2
}{
\|\sqrt{\eta_i}z\|^2
}
$$

展开：

$$
R_i
=
\frac{
\eta_i^2\|s_\theta(x,\sigma_i)\|^2/4
}{
\eta_i\|z\|^2
}
$$

得到：

$$
\boxed{
R_i
=
\frac{\eta_i}{4}
\frac{
\|s_\theta(x,\sigma_i)\|^2
}{
\|z\|^2
}
}
$$

前面提到，作者经验上观察到：

$$
\|s_\theta(x,\sigma_i)\|
\propto
\frac1{\sigma_i}
$$

因此：

$$
\|s_\theta(x,\sigma_i)\|^2
\propto
\frac1{\sigma_i^2}
$$

所以：

$$
R_i\propto\frac{\eta_i}{\sigma_i^2}
$$

如果希望：

$$
R_i
$$

不随噪声尺度明显变化，就需要：

$$
\frac{\eta_i}{\sigma_i^2}
=
\text{常数}
$$

于是：

$$
\boxed{
\eta_i\propto\sigma_i^2
}
$$

这就是论文采用该步长调度的原因。

它并不是严格的最优步长定理，而是为了平衡不同噪声尺度下的 Score 引导和随机探索。

---

## 8.4 Algorithm 1 的完整执行过程

给定：

$$
\{\sigma_i\}_{i=1}^{L}
$$

基础步长：

$$
\varepsilon
$$

每个尺度的迭代次数：

$$
K
$$

首先初始化：

$$
x\sim p_{\mathrm{init}}
$$

然后依次遍历：

$$
i=1,\ldots,L
$$

计算：

$$
\boxed{
\eta_i
=
\varepsilon
\frac{\sigma_i^2}{\sigma_L^2}
}
$$

对于每个噪声尺度，重复：

$$
k=1,\ldots,K
$$

采样：

$$
z_k\sim\mathcal N(0,I)
$$

更新：

$$
\boxed{
x\leftarrow
x+
\frac{\eta_i}{2}s_\theta(x,\sigma_i)
+
\sqrt{\eta_i}z_k
}
$$

完成所有尺度后，返回最终样本：

$$
x_{\mathrm{generated}}
$$

这就是论文 Algorithm 1。

### 需要注意的一个细节

论文没有在每个噪声尺度切换时，额外执行一次显式去噪操作。

也就是说，从：

$$
\sigma_i
$$

切换到：

$$
\sigma_{i+1}
$$

时，只是改变：

$$
s_\theta(x,\sigma_i)
\rightarrow
s_\theta(x,\sigma_{i+1})
$$

以及 Langevin 步长：

$$
\eta_i\rightarrow\eta_{i+1}
$$

当前样本：

$$
x
$$

直接作为下一阶段的初始化。

整个生成过程依靠 Langevin Dynamics 自身逐渐调整样本分布。

---

# 九、NCSN 的网络结构：论文附录 A

论文没有采用简单的 MLP，而是针对图像生成设计了一种多尺度卷积网络。

作者认为，Score 网络的输出与输入图像具有相同维度，因此可以借鉴图像密集预测任务的网络结构。

整体采用基于 **RefineNet** 的结构，并结合：

- U-Net 风格的多尺度特征融合；
- Dilated Convolution（空洞卷积）；
- Conditional Instance Normalization；
- 作者改进的 CondInstanceNorm++。

其中最值得理解的是噪声条件如何注入网络。

## 9.1 Conditional Instance Normalization

假设某一层特征图为：

$$
h\in\mathbb R^{C\times H\times W}
$$

其中：

- $C$：通道数。
- $H,W$：空间分辨率。

对于第 $k$ 个通道：

$$
h_k\in\mathbb R^{H\times W}
$$

计算其空间均值：

$$
\mu_k
=
\frac1{HW}
\sum_{u=1}^{H}
\sum_{v=1}^{W}
h_{k,u,v}
$$

以及标准差：

$$
d_k
=
\sqrt{
\frac1{HW}
\sum_{u,v}(h_{k,u,v}-\mu_k)^2
}
$$

普通 Instance Normalization 为：

$$
\widehat h_k
=
\frac{h_k-\mu_k}{d_k}
$$

而 Conditional Instance Normalization 为：

$$
\boxed{
z_k
=
\gamma[i,k]
\frac{h_k-\mu_k}{d_k}
+
b[i,k]
}
$$

其中：

- $i$：噪声尺度索引。
- $k$：特征通道索引。
- $\gamma[i,k]$：对应噪声尺度和通道的可学习缩放参数。
- $b[i,k]$：对应噪声尺度和通道的可学习偏置。

因此，不同噪声尺度：

$$
\sigma_i
$$

会使用不同的归一化参数。

这样，网络就可以根据噪声尺度调整中间特征。

## 9.2 为什么作者又提出 CondInstanceNorm++？

普通 Instance Normalization 存在一个问题：

$$
\widehat h_k
=
\frac{h_k-\mu_k}{d_k}
$$

它会去掉每个通道的均值信息：

$$
\mu_k
$$

但在图像生成任务中，不同通道的均值可能包含颜色、亮度等信息。

作者观察到，直接使用 Instance Normalization 可能导致生成图像出现颜色偏移。

因此提出 CondInstanceNorm++。

首先计算所有通道均值的均值：

$$
m=\frac1C\sum_{k=1}^{C}\mu_k
$$

以及这些均值的标准差：

$$
v=
\sqrt{
\frac1C\sum_{k=1}^{C}(\mu_k-m)^2
}
$$

然后加入一个新的条件项：

$$
\boxed{
z_k
=
\gamma[i,k]
\frac{h_k-\mu_k}{d_k}
+
b[i,k]
+
a[i,k]\frac{\mu_k-m}{v}
}
$$

其中：

$$
a[i,k]
$$

也是可学习参数。

实际实现时需要在标准差分母中加入适当的数值稳定项。

相比普通 Conditional Instance Normalization，新增加的：

$$
a[i,k]\frac{\mu_k-m}{v}
$$

能够保留不同通道均值之间的相对信息。

这有助于缓解图像颜色偏移。

## 9.3 网络整体设计

根据附录 A，作者使用四级级联 RefineNet，并进行以下修改：

- 使用预激活残差块。
- 将 Batch Normalization 替换为 CondInstanceNorm++。
- 使用 Average Pooling 替代部分 Max Pooling。
- 使用空洞卷积扩大感受野。
- 使用 ELU 激活函数。
- 通过多尺度特征融合保留图像空间结构。

对于 CelebA 和 CIFAR-10，第一阶段使用 128 个特征通道，后续阶段使用更多通道；MNIST 使用更小的网络宽度。

网络最终输出与输入图像形状一致的 Score：

$$
\boxed{
s_\theta(x,\sigma_i)\in\mathbb R^{C\times H\times W}
}
$$

这里需要特别注意：**NCSN 的条件输入不是通过简单地把噪声标准差拼接到图像通道上实现的，而是主要通过条件归一化参数影响网络特征。**[arXiv](https://arxiv.org/pdf/1907.05600?utm_source=chatgpt.com)

---

# 十、论文实验：作者如何验证方法有效？

对应正文第 5 节及附录 B、C。

论文的实验并不只是展示生成图像，还包括对两个核心理论动机的验证。

## 10.1 实验一：高斯噪声是否改善 Score Matching 的稳定性？

对应 Figure 1。

作者在 CIFAR-10 上训练 Score 网络，使用 Sliced Score Matching。

对比两种情况：

**情况 A：不添加噪声。**

直接使用原始图像训练。

**情况 B：添加微小高斯噪声。**

$$
\widetilde x=x+\epsilon
$$

其中：

$$
\epsilon\sim\mathcal N(0,0.0001I)
$$

实验观察到：

- 原始数据上的训练损失存在明显波动。
- 加噪数据上的训练损失更加稳定。

作者认为，这与真实数据可能集中在低维流形附近有关。

高斯扰动使训练分布具有全空间支撑，从而改善 Score 估计。

这一实验主要验证的是：**即使噪声非常小，也可能显著改善 Score Matching 的训练性质。**

---

## 10.2 实验二：低概率区域中的 Score 估计误差

对应 Figure 2。

作者构造二维高斯混合分布：

$$
\boxed{
p(x)
=
0.2\mathcal N((-5,-5),I)
+
0.8\mathcal N((5,5),I)
}
$$

使用一个三层 MLP 估计 Score。

因为该分布具有解析表达式，所以真实 Score 可以直接计算。

作者比较：

$$
s^*(x)=\nabla\log p(x)
$$

和：

$$
s_\theta(x)
$$

的向量场。

实验结果表明：

在两个高概率模态附近：

$$
s_\theta(x)\approx s^*(x)
$$

但在两个模态之间以及远离数据的区域，预测误差明显增大。

这说明：

**即使 Score Matching 的总体训练损失较低，也不意味着模型在整个空间中都具有准确的 Score 估计。**

因为损失是对：

$$
p_{\mathrm{data}}(x)
$$

取期望。

低概率区域对损失的贡献较小。

---

## 10.3 实验三：Annealed Langevin Dynamics 是否改善模态混合？

对应 Figure 3。

仍然使用二维高斯混合分布。

作者比较三种结果：

1. 直接从真实混合高斯分布采样。
2. 使用普通 Langevin Dynamics。
3. 使用 Annealed Langevin Dynamics。

这里有一个重要设计：

**普通 Langevin 和退火 Langevin 都使用真实 Score，而不是神经网络预测的 Score。**

因此可以排除神经网络估计误差的影响。

实验发现：

普通 Langevin Dynamics 没有正确恢复两个模态的相对概率质量。

而 Annealed Langevin Dynamics 的采样结果更接近真实分布。

这说明退火采样改善的不只是 Score 网络的预测条件，还包括采样过程本身的模态混合。

需要说明一个原文细节：正文第 4.3 节和附录 B.1 对这个 Toy Experiment 的具体噪声范围记载并不完全一致。正文给出 $L=10,\sigma_1=10,\sigma_{10}=0.1$，附录 B.1 则给出 $L=10,\sigma_1=20,\sigma_{10}=1$。因此复现实验时应以实际代码和配置进一步核对，不能将其中一组数值当作毫无歧义的最终设置。[arXiv](https://arxiv.org/pdf/1907.05600?utm_source=chatgpt.com)

---

## 10.4 实验四：图像生成质量

论文在三个数据集上测试：

- MNIST；
- CelebA；
- CIFAR-10。

图像像素统一缩放至：

$$
[0,1]
$$

噪声尺度数量：

$$
\boxed{L=10}
$$

最大噪声标准差：

$$
\boxed{\sigma_1=1}
$$

最小噪声标准差：

$$
\boxed{\sigma_{10}=0.01}
$$

噪声尺度采用几何级数。

采样时，每个噪声尺度执行：

$$
\boxed{K=100}
$$

次 Langevin 更新。

基础步长：

$$
\boxed{\varepsilon=2\times10^{-5}}
$$

因此总共需要：

$$
\boxed{L\times K=1000}
$$

次网络调用。

### 训练设置

根据附录 B.2：

| 参数 | 设置 |
|---|---|
| 优化器 | Adam |
| 学习率 | 0.001 |
| Batch Size | 128 |
| 训练迭代次数 | 200,000 |
| Checkpoint 间隔 | 5,000 次 |
| 数据增强 | CelebA、CIFAR-10 随机翻转 |
| 噪声尺度 | 10 个，几何级数 |
| 采样总步数 | 1000 |

CIFAR-10 和 CelebA 的模型通过生成样本的 FID 选择 Checkpoint。

### CIFAR-10 结果

论文报告：

$$
\boxed{
\text{Inception Score}=8.87\pm0.12
}
$$

$$
\boxed{
\text{FID}=25.32
}
$$

这说明该方法在当时能够生成质量较高的图像。

但要注意，Inception Score 和 FID 衡量的是生成样本的统计性质，不能单独证明模型准确恢复了真实概率密度。[arXiv](https://arxiv.org/pdf/1907.05600?utm_source=chatgpt.com)

---

## 10.5 实验五：单噪声尺度为什么不够？

这是论文中一个重要的对照实验。

作者训练了一个 Baseline：

$$
\boxed{
\sigma=0.01
}
$$

即只使用一个很小的噪声尺度。

该方法仍然能够通过高斯扰动避免原始数据严格位于低维流形的问题。

但它不能充分解决低概率区域中的 Score 估计问题。

因为：

$$
\sigma=0.01
$$

太小，扰动后的分布仍然非常接近原始数据分布。

因此，两个模态之间的低概率区域仍然缺少足够的训练信号。

采样时也仍然面临混合缓慢的问题。

实验结果表明，单噪声尺度模型无法生成质量合理的图像。

而多噪声尺度 NCSN 能够成功生成 MNIST、CelebA 和 CIFAR-10 图像。

这个实验验证了：

$$
\boxed{
\text{仅仅添加高斯噪声还不够，关键是使用多个噪声尺度并进行退火采样。}
}
$$

不过，这个对照同时改变了训练噪声尺度设置和采样方式，因此它验证的是完整方案相对于单尺度基线的优势，不能单独量化每个组成部分的贡献。[arXiv](https://arxiv.org/pdf/1907.05600?utm_source=chatgpt.com)

---

# 十一、图像修复：Algorithm 2

论文还展示了 NCSN 的图像修复能力。

这部分对应正文 Figure 6 和附录 B.3。

假设我们有一张原始图像：

$$
x
$$

但只有部分像素可见。

定义掩码：

$$
m\in\{0,1\}^{D}
$$

其中：

$$
m_j=
\begin{cases}
1,&\text{第 }j\text{ 个像素已知}\\
0,&\text{第 }j\text{ 个像素缺失}
\end{cases}
$$

目标是在保留已知区域的同时，生成合理的缺失区域。

## 11.1 为什么可以使用 NCSN 修复图像？

因为 NCSN 学习的是整个图像分布的 Score：

$$
s_\theta(x,\sigma_i)
$$

它描述的是完整图像在概率空间中的变化方向。

因此，可以使用 Langevin Dynamics 对整张图像进行采样，同时强制已知像素满足观测约束。

## 11.2 论文的修复算法

对于某个噪声尺度：

$$
\sigma_i
$$

首先对已知图像添加同尺度高斯噪声：

$$
\boxed{
y=x+\sigma_i\epsilon
}
$$

其中：

$$
\epsilon\sim\mathcal N(0,I)
$$

然后执行 Langevin 更新：

$$
\boxed{
\widetilde x_{k+1}
=
\widetilde x_k
+
\frac{\eta_i}{2}s_\theta(\widetilde x_k,\sigma_i)
+
\sqrt{\eta_i}z_k
}
$$

接着，将已知区域替换为加噪观测：

$$
\boxed{
\widetilde x_{k+1}
\leftarrow
(1-m)\odot\widetilde x_{k+1}
+
m\odot y
}
$$

其中：

$$
\odot
$$

表示逐元素乘法。

因此：

- 缺失区域由 Langevin Dynamics 生成。
- 已知区域由观测图像约束。
- 随着噪声尺度降低，已知区域也逐渐接近原始观测。

为什么不直接把已知区域固定为干净图像？

因为当前 Score 网络学习的是：

$$
q_{\sigma_i}(x)
$$

即噪声尺度为 $\sigma_i$ 的扰动分布。

如果在大噪声阶段直接固定干净像素，可能导致已知区域与当前噪声尺度下的其他像素不一致。

因此作者先构造：

$$
y=x+\sigma_i\epsilon
$$

使已知区域与当前扰动分布的噪声尺度一致。

随着：

$$
\sigma_i\rightarrow0
$$

这些已知像素也逐渐接近原始图像。

论文在 CelebA 和 CIFAR-10 上展示了不同形状遮挡区域的修复结果。

由于 Langevin Dynamics 包含随机噪声，同一个遮挡图像还可以生成不同的合理修复结果。

这说明模型不仅能够生成完整图像，还学习到了具有一定泛化能力的图像分布结构。[arXiv](https://arxiv.org/pdf/1907.05600?utm_source=chatgpt.com)

---

# 十二、论文的理论边界与局限

通读这篇论文后，我认为有几个地方需要特别准确地理解。

### 12.1 DSM 的最优解是边缘 Score，不是单次噪声

训练监督目标：

$$
-\frac{\epsilon}{\sigma}
$$

依赖当前采样的干净数据和噪声。

但网络输入只有：

$$
\widetilde x,\sigma
$$

所以最优预测为：

$$
\boxed{
s_\theta^*(\widetilde x,\sigma)
=
\mathbb E
\left[
-\frac{\epsilon}{\sigma}
\;\middle|\;
\widetilde x
\right]
}
$$

也就是：

$$
\boxed{
\nabla_{\widetilde x}\log q_\sigma(\widetilde x)
}
$$

这是严格的数学结论。

### 12.2 $\lambda(\sigma)=\sigma^2$ 是尺度平衡设计

作者选择：

$$
\lambda(\sigma)=\sigma^2
$$

是为了让不同噪声尺度的损失贡献具有相近的数量级。

它不是通过某个唯一的最优性定理得到的。

对于有限容量网络，不同权重可能产生不同的训练效果。

### 12.3 Langevin Dynamics 的平稳分布不等于有限步采样分布

连续 Langevin SDE 在适当条件下具有目标平稳分布：

$$
p(x)
$$

但论文实际使用有限步长：

$$
\eta>0
$$

以及有限次迭代：

$$
K<\infty
$$

因此：

$$
p_{\mathrm{generated}}
$$

通常只能近似目标分布。

### 12.4 退火采样改善混合，但没有普遍的有限步精确性保证

作者通过 Toy Experiment 和图像生成实验说明：

$$
\text{Annealed Langevin Dynamics}
$$

比普通 Langevin Dynamics 更有效。

但论文没有证明，对于任意多模态分布和任意有限采样步数，退火 Langevin 都一定能够恢复正确的模态比例。

它的优势依赖噪声尺度、步长、每个尺度的迭代次数，以及 Score 估计的准确性。

### 12.5 训练与采样是两个独立环节

NCSN 的训练目标是：

$$
\mathcal L_{\mathrm{DSM}}
$$

而推理使用：

$$
\text{Annealed Langevin Dynamics}
$$

训练过程中不需要运行 Langevin 采样链。

这意味着 Score 估计方法和采样方法在概念上是分离的。

这是论文强调的一个重要特点。

---

# 十三、整篇论文总结

这篇论文的研究逻辑非常完整。

首先，作者提出一个新的生成建模思路：

$$
\boxed{
\text{不直接学习概率密度，而是学习概率密度的 Score。}
}
$$

然后通过 Langevin Dynamics，证明仅依赖 Score 就能够构造具有目标平稳分布的随机采样过程。

但直接将该方法应用到真实数据时，会遇到两个问题：

$$
\boxed{
\text{低维流形导致 Score 难以估计}
}
$$

以及：

$$
\boxed{
\text{低概率区域缺少训练信号，且采样混合缓慢}
}
$$

为此，作者引入多个高斯扰动分布：

$$
\boxed{
q_{\sigma_i}(x)
=
\int
p_{\mathrm{data}}(x_0)
\mathcal N(x;x_0,\sigma_i^2I)dx_0
}
$$

通过 Denoising Score Matching，训练一个条件网络：

$$
\boxed{
s_\theta(x,\sigma_i)
\approx
\nabla_x\log q_{\sigma_i}(x)
}
$$

最终训练目标为：

$$
\boxed{
\mathcal L(\theta)
=
\frac1{2L}
\sum_{i=1}^{L}
\mathbb E_{x_0,\epsilon}
\left[
\left\|
\sigma_i s_\theta(x_0+\sigma_i\epsilon,\sigma_i)
+
\epsilon
\right\|^2
\right]
}
$$

推理时，从随机噪声初始化，并逐渐降低噪声尺度：

$$
\boxed{
\sigma_1>\sigma_2>\cdots>\sigma_L
}
$$

每个噪声尺度执行：

$$
\boxed{
x_{k+1}
=
x_k+
\frac{\eta_i}{2}s_\theta(x_k,\sigma_i)
+
\sqrt{\eta_i}z_k
}
$$

其中：

$$
\boxed{
\eta_i
=
\varepsilon\frac{\sigma_i^2}{\sigma_L^2}
}
$$

最终得到接近真实数据分布的样本。

**这篇论文最重要的三个数学结论是：**

**第一，Score Matching 可以通过分部积分消去未知的真实 Score：**

$$
\boxed{
\mathcal J_{\mathrm{SM}}
=
\mathbb E_p
\left[
\frac12\|s_\theta\|^2
+
\nabla\cdot s_\theta
\right]
}
$$

**第二，DSM 可以通过条件高斯 Score 学习加噪边缘分布的 Score：**

$$
\boxed{
\nabla\log q_\sigma(\widetilde x)
=
\mathbb E
\left[
-\frac{\widetilde x-x}{\sigma^2}
\;\middle|\;
\widetilde x
\right]
}
$$

**第三，Langevin Dynamics 可以利用 Score 构造以目标分布为平稳分布的随机过程：**

$$
\boxed{
dX_\tau
=
\frac12\nabla\log p(X_\tau)d\tau+dW_\tau
}
$$

从而通过离散化得到实际采样算法。

最后，作者将这三个理论基础与多噪声尺度训练和退火采样结合起来，形成完整的 NCSN 生成模型。

**整篇论文真正的创新点，是利用多尺度高斯扰动同时改善 Score 的可学习性和 Langevin 采样的混合性质，而不是单独提出 Score Matching 或 Langevin Dynamics。**

这也解释了为什么论文的核心实验都围绕两个问题展开：**Score 能否在低概率区域被准确学习，以及 Langevin Dynamics 能否在多模态分布之间有效混合。**